# 煙霧偵測模型訓練 — Google Colab

**資料集：** [Roboflow smoke-gxoy3](https://universe.roboflow.com/naruesuan-university/smoke-gxoy3) + 自動合併額外公開資料集  
**模型：** YOLOv8s  
**目標：** 訓練後匯出 ONNX，放到 `models/smoke_detector.onnx` 啟用條件 3

**開始前：** 上方選單 → `執行階段 → 變更執行階段類型 → T4 GPU`

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. 安裝套件

In [ ]:
!pip install -q roboflow ultralytics onnx onnxsim

## 2. 下載主資料集（smoke-gxoy3）

填入你的 Roboflow API Key：[取得方式](https://app.roboflow.com/) → Settings → Roboflow API

In [ ]:
from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"  # ← 替換為你的 Key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("naruesuan-university").project("smoke-gxoy3")

versions = project.versions()
print(f"可用版本：{[v.version for v in versions]}")
latest = versions[-1]
print(f"使用版本：{latest.version}")

dataset = latest.download("yolov8", location="/content/smoke_dataset")
print(f"下載完成：{dataset.location}")

## 3. 搜尋並合併額外公開煙霧資料集

In [ ]:

# 使用 Roboflow SDK 搜尋公開煙霧資料集
search_results = []

try:
    # SDK 內建搜尋
    results = rf.search_datasets(query="smoke", type="project")
    for r in results.get("results", []):
        ws   = r.get("workspace_url") or r.get("workspace") or ""
        proj = r.get("slug") or r.get("project") or ""
        cnt  = r.get("images", 0)
        if ws and proj:
            search_results.append(r)
            print(f"  {ws}/{proj}  ({cnt} 張)")
    print(f"找到 {len(search_results)} 個結果")
except Exception as e:
    print(f"SDK 搜尋失敗：{e}")

# 若搜尋失敗或結果太少，補充已知可用的公開資料集
FALLBACK_DATASETS = [
    ("agentmakerpro",  "smoke-dataset"),
    ("david-lee-d0sbc","smoke-detection"),
    ("fire-and-smoke", "fire-smoke-tki2x"),
    ("fy-4bppb",       "smoke-detection-s6jee"),
    ("smoke-v2xkm",    "smoke-v2xkm"),
]

if len(search_results) < 3:
    print("\n補充已知公開資料集（若不存在會自動跳過）：")
    for ws, proj in FALLBACK_DATASETS:
        print(f"  {ws}/{proj}")
    # 轉為 search_results 格式
    for ws, proj in FALLBACK_DATASETS:
        search_results.append({"workspace_url": ws, "slug": proj, "images": 999})


In [ ]:
import os, shutil, glob, random

MERGED_DIR   = "/content/smoke_merged"
MAX_DATASETS = 5
MIN_IMAGES   = 100

def detect_split_dirs(base):
    for img_cand in ["train/images", "images/train", "train"]:
        imgs = glob.glob(f"{base}/{img_cand}/*.jpg") + glob.glob(f"{base}/{img_cand}/*.png")
        if imgs:
            for lbl_cand in ["train/labels", "labels/train"]:
                if glob.glob(f"{base}/{lbl_cand}/*.txt"):
                    return img_cand, lbl_cand
    return None, None

def copy_to_merged(src_base, img_rel, lbl_rel):
    for kind, rel in [("images", img_rel), ("labels", lbl_rel)]:
        src = os.path.join(src_base, rel)
        dst = os.path.join(MERGED_DIR, kind, "train")
        os.makedirs(dst, exist_ok=True)
        files = glob.glob(f"{src}/*")
        for f in files:
            d = os.path.join(dst, os.path.basename(f))
            if os.path.exists(d):
                stem, ext = os.path.splitext(os.path.basename(f))
                d = os.path.join(dst, f"{stem}_{random.randint(1000,9999)}{ext}")
            shutil.copy2(f, d)
        print(f"    {kind}: {len(files)} 個")

os.makedirs(MERGED_DIR, exist_ok=True)

# 複製主資料集
print("=== 主資料集 smoke-gxoy3 ===")
img_rel, lbl_rel = detect_split_dirs("/content/smoke_dataset")
if img_rel:
    copy_to_merged("/content/smoke_dataset", img_rel, lbl_rel)
else:
    print("  ⚠️ 找不到主資料集，請先執行 Cell 2")

# 從搜尋結果下載額外資料集
downloaded = 0
skip = {"smoke-gxoy3"}

for r in search_results:
    if downloaded >= MAX_DATASETS:
        break
    ws   = r.get("workspace_url") or r.get("workspace") or ""
    proj = r.get("slug") or r.get("project") or ""
    cnt  = r.get("images", 0)
    if not ws or not proj or proj in skip or cnt < MIN_IMAGES:
        continue
    print(f"\n=== {ws}/{proj} ({cnt} 張) ===")
    try:
        p = rf.workspace(ws).project(proj)
        vers = p.versions()
        if not vers:
            continue
        dl_dir = f"/content/extra_{proj}"
        vers[-1].download("yolov8", location=dl_dir)
        img_rel, lbl_rel = detect_split_dirs(dl_dir)
        if img_rel:
            copy_to_merged(dl_dir, img_rel, lbl_rel)
            skip.add(proj)
            downloaded += 1
        else:
            print("  ⚠️ 無法偵測目錄結構")
    except Exception as e:
        print(f"  ❌ {e}")

total = glob.glob(f"{MERGED_DIR}/images/train/*.jpg") + glob.glob(f"{MERGED_DIR}/images/train/*.png")
print(f"\n✅ 合併完成：共 {len(total)} 張（額外 {downloaded} 個資料集）")

## 4. 切割驗證集 & 建立 data.yaml

In [ ]:
import yaml, random, shutil, glob, os

MERGED_DIR = "/content/smoke_merged"

all_imgs = sorted(
    glob.glob(f"{MERGED_DIR}/images/train/*.jpg") +
    glob.glob(f"{MERGED_DIR}/images/train/*.png")
)
random.seed(42)
random.shuffle(all_imgs)
split = max(10, int(len(all_imgs) * 0.1))

os.makedirs(f"{MERGED_DIR}/images/val", exist_ok=True)
os.makedirs(f"{MERGED_DIR}/labels/val", exist_ok=True)

for img_path in all_imgs[:split]:
    fname = os.path.basename(img_path)
    stem  = os.path.splitext(fname)[0]
    shutil.move(img_path, f"{MERGED_DIR}/images/val/{fname}")
    lbl = f"{MERGED_DIR}/labels/train/{stem}.txt"
    if os.path.exists(lbl):
        shutil.move(lbl, f"{MERGED_DIR}/labels/val/{stem}.txt")

n_train = len(glob.glob(f"{MERGED_DIR}/images/train/*.jpg") + glob.glob(f"{MERGED_DIR}/images/train/*.png"))
n_val   = len(glob.glob(f"{MERGED_DIR}/images/val/*.jpg")   + glob.glob(f"{MERGED_DIR}/images/val/*.png"))
print(f"train={n_train} 張，val={n_val} 張")

yaml_path = f"{MERGED_DIR}/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump({
        "path":  MERGED_DIR,
        "train": "images/train",
        "val":   "images/val",
        "nc":    1,
        "names": ["smoke"],
    }, f, default_flow_style=False)

print(f"✅ data.yaml 寫入完成")
!cat /content/smoke_merged/data.yaml

## 5. 訓練模型（YOLOv8s）

T4 GPU 大約需要 **30–50 分鐘**

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print(f"使用裝置：{'GPU (' + torch.cuda.get_device_name(0) + ')' if device == 0 else 'CPU'}")

DATA_YAML = ("/content/smoke_merged/data.yaml"
             if os.path.exists("/content/smoke_merged/data.yaml")
             else "/content/smoke_dataset/data.yaml")
print(f"資料集：{DATA_YAML}")

model = YOLO("yolov8s.pt")

model.train(
    data=DATA_YAML,
    epochs=100,
    batch=16 if device == 0 else 4,
    imgsz=640,
    device=device,
    project="/content/runs",
    name="smoke_detector",
    patience=20,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    fliplr=0.5, mosaic=1.0, mixup=0.1, copy_paste=0.1,
    save=True, save_period=10, exist_ok=True,
)

## 6. 驗證指標

In [ ]:
import glob as _glob
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

candidates = sorted(_glob.glob("/content/runs/smoke_detector*/weights/best.pt"))
assert candidates, "找不到 best.pt，請確認訓練已完成"
best_pt = candidates[-1]
print(f"模型：{best_pt}")

eval_model = YOLO(best_pt)
metrics = eval_model.val(data=DATA_YAML)
print(f"\nmAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

for p in sorted(_glob.glob("/content/runs/smoke_detector*/*.png")):
    display(IPImage(p))

## 7. 匯出 ONNX

In [ ]:
from ultralytics import YOLO

YOLO(best_pt).export(format="onnx", imgsz=640, simplify=True, opset=17)
onnx_path = best_pt.replace(".pt", ".onnx")
print(f"✅ ONNX：{onnx_path}")

## 8. 下載模型

In [ ]:
import shutil, os
from google.colab import files

dst = "/content/smoke_detector.onnx"
shutil.copy(onnx_path, dst)
print(f"檔案大小：{os.path.getsize(dst)/1024/1024:.1f} MB")
files.download(dst)
print("\n下載完成後複製到 cgr_detection/models/smoke_detector.onnx")

## 9. （選用）下載 .pt 權重備份

In [ ]:
from google.colab import files
files.download(best_pt)